In [1]:
from pathlib import Path
from urllib.parse import urlparse, parse_qs
import yt_dlp
from tqdm import tqdm

1. Read video IDs or URLs from a file, one line at a time

In [2]:
def stream_video_urls(file_path):
    """Yield each non-empty, non-comment line from a text file."""

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            item = line.strip()

            # Ignore blank lines and comments.
            if item and not item.startswith("#"):
                yield item
    

In [3]:
video_urls = list(stream_video_urls("/home/ibukhari/Documents/Dev/Signet_PersonalExperiments/datasets/youtube-asl_youtube_asl_video_ids.txt"))

In [4]:
sample_index = 345
print(f"Total Videos: {len(video_urls)}")
print(f"Video URL at index {sample_index}:   {video_urls[sample_index]}")

Total Videos: 11096
Video URL at index 345:   N4TPKucBOyU


2. Extract the YouTube video ID from an input (extra check but not needed as txt file is clean)

In [5]:
def get_video_id(value):
    """Accept a YouTube video ID or a supported YouTube URL."""

    value = str(value).strip()

    if not value:
        return None

    # A plain ID does not contain a URL scheme or path.
    if "://" not in value and "/" not in value:
        return value

    parsed = urlparse(value)
    host = (parsed.hostname or "").lower()

    # Example: https://youtu.be/VIDEO_ID
    if host == "youtu.be":
        return parsed.path.strip("/").split("/")[0] or None

    # Examples: youtube.com/watch?v=VIDEO_ID
    #            youtube.com/shorts/VIDEO_ID
    #            youtube.com/embed/VIDEO_ID
    if host == "youtube.com" or host.endswith(".youtube.com"):
        if parsed.path == "/watch":
            return parse_qs(parsed.query).get("v", [None])[0]

        parts = parsed.path.strip("/").split("/")

        if len(parts) >= 2 and parts[0] in {
            "embed", "shorts", "live", "v"
        }:
            return parts[1]

    return None

3. Download Videos and Subs

In [6]:
# ---------------------------------------------------------
# 1. INITIALISE YOUTUBE DOWNLOADER
# ---------------------------------------------------------


def initialize_ydl(
    video_dir,
    subtitle_dir,
    verbose=False,
    **option_overrides
):

    options = {
        "format": (
            "bestvideo[ext=mp4]+bestaudio[ext=m4a]/"
            "best[ext=mp4]/bestvideo+bestaudio/best"
        ),
        "merge_output_format": "mp4",

        "outtmpl": {
            "default": str(video_dir / "%(id)s.%(ext)s"),
            "subtitle": str(subtitle_dir / "%(id)s.%(ext)s"),
        },

        "paths": {
            "home": str(video_dir),
            "subtitle": str(subtitle_dir),
        },

        "writesubtitles": True,
        "writeautomaticsub": False,
        "subtitleslangs": ["en.*", "eng.*", "english.*"],

        "ffmpeg_location": "/usr/bin/ffmpeg",
        "ignoreerrors": False,
        "noplaylist": True,
        "quiet": not verbose,
        "no_warnings": not verbose,
        "noprogress": True,
    }

    # Override default options when required.
    options.update(option_overrides)

    return yt_dlp.YoutubeDL(options)


In [7]:
# ---------------------------------------------------------
# 2. CHECK WHETHER VIDEO AND SUBTITLES ALREADY EXIST
# ---------------------------------------------------------

def file_exists(directory, video_id):

    # Return immediately when a matching completed file is found.
    for path in directory.glob(f"{video_id}.*"):

        if (
            path.is_file()
            and not path.name.endswith((".part", ".ytdl", ".tmp"))
        ):
            return True

    return False


def check_existing_files(video_id, video_dir, subtitle_dir):

    video_exists = file_exists(video_dir, video_id)
    subtitle_exists = file_exists(subtitle_dir, video_id)

    return video_exists, subtitle_exists


In [8]:
# ---------------------------------------------------------
# 3. DOWNLOAD ALL AVAILABLE MANUAL SUBTITLES
# ---------------------------------------------------------

def download_manual_subtitles(
    url,
    video_id,
    video_dir,
    subtitle_dir,
    verbose=False
):

    # Initialise a separate downloader for subtitles only.
    ydl =  initialize_ydl(
        video_dir,
        subtitle_dir,
        verbose=verbose,
        subtitleslangs=["all", "-live_chat"],  # Download every available manually created subtitle.
        writeautomaticsub=False, # Do not download automatically generated subtitles.
        skip_download=True, # Skip downloading the actual video.
    )

    info = ydl.extract_info(url, download=True)

    # Retrieve the actual video ID.
    actual_id = info.get("id", video_id) if info else video_id

    # Check whether subtitles were successfully downloaded.
    _, subtitle_exists = check_existing_files(
        actual_id,
        video_dir,
        subtitle_dir
    )

    return actual_id, subtitle_exists

In [9]:
# ---------------------------------------------------------
# 4. REMOVE VIDEO FILES
# ---------------------------------------------------------

def remove_video_files(video_id, video_dir):

    removed_files = 0

    for path in video_dir.glob(f"{video_id}.*"):

        if path.is_file():
            path.unlink()
            removed_files += 1

    return removed_files


In [10]:
# ---------------------------------------------------------
# 5. MAIN DOWNLOAD FUNCTION
# ---------------------------------------------------------

def download_youtube_videos(
    video_urls,
    video_output_path,
    subtitle_output_path,
    subtitle_langs=("en.*", "eng.*", "english.*"),
    write_auto_subs=False,
    verbose=False,
):

    # Prepare output directories.
    video_dir = Path(video_output_path).expanduser().resolve()
    subtitle_dir = Path(subtitle_output_path).expanduser().resolve()

    video_dir.mkdir(parents=True, exist_ok=True)
    subtitle_dir.mkdir(parents=True, exist_ok=True)

    # Remove empty inputs and comments.
    urls = [
        str(item).strip()
        for item in video_urls
        if str(item).strip() and not str(item).strip().startswith("#")
    ]

    # Initialise download statistics.
    stats = {
        "urls_processed": 0,
        "videos_success": 0,
        "videos_failed": 0,
        "subtitles_success": 0,
        "subtitles_failed": 0,
    }

    if not urls:
        print("No valid URLs provided.")
        return stats

    # Initialise the main downloader.
    with initialize_ydl(
        video_dir,
        subtitle_dir,
        verbose=verbose,
        subtitleslangs=list(subtitle_langs),
        writeautomaticsub=write_auto_subs,
    ) as ydl:

        # Initialise progress bar.
        with tqdm(
            total=len(urls),
            desc="Downloading",
            unit="URL",
            colour="black",
            dynamic_ncols=True,
        ) as progress:

            for url in urls:

                video_id = None
                actual_id = None

                video_success = False
                subtitle_success = False

                try:
                    # Extract and validate the video ID.
                    video_id = get_video_id(url)

                    if not video_id:
                        raise ValueError("Invalid YouTube URL or video ID")

                    actual_id = video_id

                    # Check whether video and subtitles already exist.
                    video_exists, subtitle_exists = check_existing_files(
                        actual_id,
                        video_dir,
                        subtitle_dir
                    )

                    subtitle_success = subtitle_exists

                    # Download the video if it does not already exist.
                    if not video_exists:

                        info = ydl.extract_info(url, download=True)

                        # Retrieve the actual downloaded video ID.
                        if info:
                            actual_id = info.get("id", video_id)

                        # Check the output directories again.
                        video_exists, subtitle_exists = check_existing_files(
                            actual_id,
                            video_dir,
                            subtitle_dir
                        )

                        subtitle_success = subtitle_exists

                    # Validate video availability.
                    if not video_exists:
                        raise RuntimeError("Video download failed")

                    video_success = True

                    # Retry subtitle downloading if subtitles are missing.
                    if not subtitle_exists:

                        retry_error = None

                        try:
                            actual_id, subtitle_exists = download_manual_subtitles(
                                url,
                                actual_id,
                                video_dir,
                                subtitle_dir,
                                verbose
                            )

                        except Exception as exc:
                            retry_error = str(exc)

                            # Check again in case files were created
                            # before yt-dlp raised an exception.
                            _, subtitle_exists = check_existing_files(
                                actual_id,
                                video_dir,
                                subtitle_dir
                            )

                        # Remove the video if manual subtitle downloading
                        # was unsuccessful.
                        if not subtitle_exists:

                            video_success = False

                            try:
                                remove_video_files(actual_id, video_dir)

                            except Exception as exc:
                                raise RuntimeError(
                                    f"Subtitle download failed and video "
                                    f"cleanup also failed: {exc}"
                                )

                            error_message = (
                                "No manually created subtitles available. "
                                "Video removed from library."
                            )

                            if retry_error:
                                error_message += f" Retry error: {retry_error}"

                            raise RuntimeError(error_message)

                        subtitle_success = True

                    print("\nVIDEO and SUBTITLE downloaded ;)\n")

                except Exception as exc:

                    if verbose:
                        progress.write(
                            f"[FAILED] {video_id or url}: {exc}"
                        )

                finally:

                    # Update statistics.
                    stats["urls_processed"] += 1

                    if video_success:
                        stats["videos_success"] += 1
                    else:
                        stats["videos_failed"] += 1

                    if subtitle_success:
                        stats["subtitles_success"] += 1
                    else:
                        stats["subtitles_failed"] += 1

                    # Update progress bar with live statistics.
                    progress.set_postfix({
                        "Video OK": stats["videos_success"],
                        "Video Fail": stats["videos_failed"],
                        "Sub OK": stats["subtitles_success"],
                        "Sub Fail": stats["subtitles_failed"],
                    })
                    
                    print("\n=====================================\n")

    # Print final statistics.
    print("\nDownload Summary")
    print("----------------")
    print(f"URLs processed:     {stats['urls_processed']}/{len(urls)}")
    print(f"Videos successful:  {stats['videos_success']}")
    print(f"Videos failed:      {stats['videos_failed']}")
    print(f"Subtitles success:  {stats['subtitles_success']}")
    print(f"Subtitles failed:   {stats['subtitles_failed']}")

    return stats

Run the workflow

In [11]:
# Run Downloads
report_path = download_youtube_videos(
    video_urls=video_urls[:20],
    video_output_path="datasets/Videos",
    subtitle_output_path="datasets/Subtitles",
    subtitle_langs=("en",),
    write_auto_subs=False,
    verbose=True,
)


Downloading:   0%|          | 0/20 [00:00<?, ?URL/s, Video OK=4, Video Fail=0, Sub OK=4, Sub Fail=0]


VIDEO and SUBTITLE downloaded ;)




VIDEO and SUBTITLE downloaded ;)




VIDEO and SUBTITLE downloaded ;)




VIDEO and SUBTITLE downloaded ;)



[youtube] Extracting URL: eOJfTd3nybA
[youtube] eOJfTd3nybA: Downloading webpage


[youtube] eOJfTd3nybA: Downloading visionos player API JSON


ERROR: [youtube] eOJfTd3nybA: This video is unavailable
Downloading:   0%|          | 0/20 [00:00<?, ?URL/s, Video OK=4, Video Fail=1, Sub OK=4, Sub Fail=1]

[FAILED] eOJfTd3nybA: ERROR: [youtube] eOJfTd3nybA: This video is unavailable


[youtube] Extracting URL: LSmWzwmlnxc
[youtube] LSmWzwmlnxc: Downloading webpage


[youtube] LSmWzwmlnxc: Downloading visionos player API JSON


... (37 outputs left out)

[youtube] MeXD-8q_HnE: Downloading visionos player API JSON
[youtube] MeXD-8q_HnE: Downloading m3u8 information
[info] MeXD-8q_HnE: Downloading subtitles: en-US
[info] MeXD-8q_HnE: Downloading 1 format(s): 136+140


[info] Writing video subtitles to: /home/ibukhari/Documents/Dev/Signet_PersonalExperiments/datasets/Videos/MeXD-8q_HnE.en-US.vtt
[download] Destination: /home/ibukhari/Documents/Dev/Signet_PersonalExperiments/datasets/Videos/MeXD-8q_HnE.en-US.vtt
[download] Download completed
[MoveFiles] Moving file "/home/ibukhari/Documents/Dev/Signet_PersonalExperiments/datasets/Videos/MeXD-8q_HnE.en-US.vtt" to "/home/ibukhari/Documents/Dev/Signet_PersonalExperiments/datasets/Subtitles/MeXD-8q_HnE.en-US.vtt"


Downloading:   0%|          | 0/20 [01:43<?, ?URL/s, Video OK=15, Video Fail=5, Sub OK=15, Sub Fail=5]


VIDEO and SUBTITLE downloaded ;)




Download Summary
----------------
URLs processed:     20/20
Videos successful:  15
Videos failed:      5
Subtitles success:  15
Subtitles failed:   5
